In [ ]:
# Windows + Jupyter asyncio fix for Playwright. Playwright launches a subprocess,
# which needs the Proactor event loop on Windows; nest_asyncio lets the notebook's
# already-running loop host Playwright's async calls. Run this cell FIRST.
import sys
import asyncio

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

import nest_asyncio
nest_asyncio.apply()

print("asyncio configured for Playwright on", sys.platform)

In [ ]:
# ============================================================
# CONFIG  ->  edit these before each run
# ============================================================

# Epidemiological year to export (Ano Epidemiologico). Typing it + Tab autocompletes
# Semana Inicial=1 / Semana Final=52 (the full year).
ANO = "2024"

# Ficha types to export, by their <select> value. The notebook loops over all of them,
# producing one file per type.  1 = SG,  2 = SRAG UTI,  3 = SRAG Hospitalizado.
TIPOS_FICHA = {"3": "SRAG_Hospitalizado", "1": "SG"}

# Tick 'Exportar dados do paciente' on the form.
EXPORTAR_DADOS_PACIENTE = True

# False = visible browser (debug / first runs).  True = invisible (unattended).
HEADLESS = False

# Slow each Playwright action down by this many ms (helps watch + debug in headed mode).
SLOW_MO_MS = 250

# How long (seconds) to wait for an export to reach 'Processamento Concluido'.
PROCESSING_TIMEOUT_S = 900

In [ ]:
# Project paths and environment setup.  Point Playwright at the project-local
# browser folder so the whole project stays portable between machines.
import os
from pathlib import Path

PROJECT_DIR = Path.cwd()
BROWSERS_DIR = PROJECT_DIR / ".playwright-browsers"
DOWNLOADS_DIR = PROJECT_DIR / "downloads"
STATE_DIR = PROJECT_DIR / "state"
LOGS_DIR = PROJECT_DIR / "logs"
STATE_FILE = STATE_DIR / "storage_state.json"

for d in (DOWNLOADS_DIR, STATE_DIR, LOGS_DIR):
    d.mkdir(exist_ok=True)

if BROWSERS_DIR.exists():
    os.environ["PLAYWRIGHT_BROWSERS_PATH"] = str(BROWSERS_DIR)

print("Project dir   :", PROJECT_DIR)
print("Browsers dir  :", BROWSERS_DIR, "(exists:", BROWSERS_DIR.exists(), ")")
print("Downloads dir :", DOWNLOADS_DIR)
print("Session file  :", STATE_FILE, "(exists:", STATE_FILE.exists(), ")")

In [ ]:
# Load credentials from the git-ignored .env file (never hard-code them here).
from dotenv import load_dotenv

load_dotenv(PROJECT_DIR / ".env")
LOGIN = os.environ["SIVEP_LOGIN"]
SENHA = os.environ["SIVEP_SENHA"]

LOGIN_URL = "https://sivepgripe.saude.gov.br/sivepgripe/login.html?1"
PRINCIPAL_URL = "https://sivepgripe.saude.gov.br/sivepgripe/visao/pages/principal.html?1"

print("Logging in as:", LOGIN)

In [ ]:
# Simple logger -> writes to logs/run_<timestamp>.log and prints to the notebook.
import logging
from datetime import datetime

_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
_log_file = LOGS_DIR / f"run_{_run_id}.log"

logger = logging.getLogger("sivep")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s")
for _h in (logging.FileHandler(_log_file, encoding="utf-8"), logging.StreamHandler()):
    _h.setFormatter(_fmt)
    logger.addHandler(_h)

def log(msg):
    logger.info(msg)

log(f"Log file: {_log_file}")

In [ ]:
# Helper functions for the SIVEP Wicket UI.  Two quirks this handles:
#   1) a startup 'Alerta' modal (ui-widget-overlay) that blocks the page
#   2) a loading spinner (div#div-carregando) shown after AJAX / submit
# Both must be cleared before the next click, or Playwright times out.

async def settle(page):
    # Dismiss the info modal if visible, then wait for overlay + spinner to vanish.
    ok = page.locator(".ui-dialog button:has-text('Ok'), button:has-text('Ok')").first
    if await ok.count() and await ok.is_visible():
        await ok.click()
        await page.wait_for_timeout(600)
    for sel in (".ui-widget-overlay", "#div-carregando"):
        try:
            await page.wait_for_selector(sel, state="hidden", timeout=8000)
        except Exception:
            pass

async def open_export_item(page, label):
    # Hover the EXPORTACAO top menu (matched by its alt attr, accent-proof) and click
    # the visible submenu link by its accessible name.  These are stateful Wicket links
    # so they must be clicked live, not navigated to by href.
    await settle(page)
    await page.locator("a.sf-with-ul[alt*='EXPORTA']").first.hover()
    await page.wait_for_timeout(900)
    link = page.get_by_role("link", name=label)
    n = await link.count()
    for i in range(n):
        if await link.nth(i).is_visible():
            await link.nth(i).click()
            break
    await page.wait_for_load_state("networkidle")
    await page.wait_for_timeout(1500)
    await settle(page)

async def listar_solicitacoes(page):
    # Return the set of 'Numero de Solicitacao' values currently in the DBF table.
    # Used to detect which row is the export we just generated.
    nums = set()
    rows = page.locator("table tr")
    for r in range(await rows.count()):
        first_cell = rows.nth(r).locator("td").first
        if await first_cell.count():
            txt = (await first_cell.inner_text()).strip()
            if txt.isdigit():
                nums.add(txt)
    return nums

print("helpers ready")

In [ ]:
# Start Playwright and open a browser context.  We use the ASYNC api because the
# notebook already runs an event loop.  If a saved session exists we load it to
# skip the login form while the cookies are still valid.
from playwright.async_api import async_playwright

_pw = await async_playwright().start()
browser = await _pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO_MS)

_ctx_kwargs = {"accept_downloads": True}
if STATE_FILE.exists():
    _ctx_kwargs["storage_state"] = str(STATE_FILE)
    log("Reusing saved session from state/storage_state.json")

context = await browser.new_context(**_ctx_kwargs)
page = await context.new_page()
log(f"Browser launched (headless={HEADLESS})")

In [ ]:
# Log in.  If the saved session is still valid the site sends us straight to the
# principal page and we skip the form.  Real field names captured from the site:
#   email -> input[name=email],  senha -> input[name=senha],  submit name=ENTRAR.
await page.goto(LOGIN_URL, wait_until="domcontentloaded")
await page.wait_for_timeout(1500)

if "login.html" in page.url:
    log("Login form present -> filling credentials")
    await page.fill("input[name='email']", LOGIN)
    await page.fill("input[name='senha']", SENHA)
    await page.click("input[type='submit'][name='ENTRAR']")
    await page.wait_for_load_state("networkidle")
    await page.wait_for_timeout(2000)
    if "login.html" in page.url:
        log("WARNING: still on login page -> check credentials")
    else:
        log(f"Login OK -> {page.url}")
        await context.storage_state(path=str(STATE_FILE))
        log("Session saved to state/storage_state.json")
else:
    log("Already authenticated via saved session")

# Dismiss the startup 'Alerta' modal (ficha-update notice) if it appears.
await settle(page)

In [ ]:
# ============================================================
# MAIN: for each ficha type -> generate an export, wait for it to finish, download.
# ============================================================
import asyncio

downloaded_files = []

for _tipo_val, _tipo_nome in TIPOS_FICHA.items():
    log(f"========== {_tipo_nome} (tipoFicha={_tipo_val}) ==========")

    # 1) Open EXPORTACAO -> CONSULTAR EXPORTACOES DBF and snapshot existing requests,
    #    so we can tell which row is the one we are about to create.
    await page.goto(PRINCIPAL_URL, wait_until="networkidle")
    await settle(page)
    await open_export_item(page, "CONSULTAR EXPORTA\u00c7\u00d5ES DBF")
    _before = await listar_solicitacoes(page)
    log(f"Existing solicitacoes before: {sorted(_before)}")

    # 2) Open REGISTROS INDIVIDUAIS and fill the form.
    await open_export_item(page, "REGISTROS INDIVIDUAIS")
    await page.select_option("select#tipoFicha", _tipo_val)
    await page.wait_for_timeout(2000)          # AJAX re-render of the period section
    await settle(page)

    # Period mode = Ano Epidemiologico; type the year and Tab to autocomplete weeks.
    await page.check("[name='periodo:anoEpidemiologico']")
    await page.wait_for_timeout(500)
    await page.fill("[name='periodo:anoAnoEpidemiologico']", ANO)
    await page.locator("[name='periodo:anoAnoEpidemiologico']").press("Tab")
    await page.wait_for_timeout(1500)
    _si = await page.input_value("[name='periodo:semanaInicial']")
    _sf = await page.input_value("[name='periodo:semanaFinal']")
    log(f"Year {ANO} -> Semana Inicial={_si}, Semana Final={_sf}")

    if EXPORTAR_DADOS_PACIENTE:
        await page.check("[name='chkExportarDadosPaciente']")

    # 3) Generate the file.
    await page.click("[name='gerarDbf']")
    await page.wait_for_load_state("networkidle")
    await page.wait_for_timeout(2000)
    await settle(page)
    log("Gerar Arquivo clicked -> export queued")

    # 4) Go to the DBF page and poll (ATUALIZAR) until the NEW request is concluded.
    await open_export_item(page, "CONSULTAR EXPORTA\u00c7\u00d5ES DBF")
    _deadline = asyncio.get_event_loop().time() + PROCESSING_TIMEOUT_S
    _new_num = None
    _dl_link = None
    while asyncio.get_event_loop().time() < _deadline:
        _now = await listar_solicitacoes(page)
        _created = sorted(_now - _before, reverse=True)
        if _created:
            _new_num = _created[0]
            # Find the row whose first cell == _new_num and check its status / link.
            _row = page.locator(f"table tr:has(td:text-is('{_new_num}'))").first
            _status = (await _row.inner_text()).lower()
            if "conclu" in _status:
                _dl_link = _row.locator("a:has-text('Download')").first
                if await _dl_link.count():
                    log(f"Solicitacao {_new_num} concluded -> downloading")
                    break
        log("Not ready yet -> ATUALIZAR in 15s")
        await page.wait_for_timeout(15000)
        _atualizar = page.locator("button:has-text('ATUALIZAR'), input[value='ATUALIZAR']").first
        if await _atualizar.count():
            await _atualizar.click()
        else:
            await page.reload(wait_until="networkidle")
        await settle(page)

    # 5) Download the file into downloads/.
    if _dl_link is not None and await _dl_link.count():
        async with page.expect_download(timeout=120000) as _dl_info:
            await _dl_link.click()
        _dl = await _dl_info.value
        _target = DOWNLOADS_DIR / f"{_run_id}_{_tipo_nome}_{ANO}_{_dl.suggested_filename}"
        await _dl.save_as(str(_target))
        downloaded_files.append(_target)
        log(f"Saved -> {_target}")
    else:
        log(f"TIMEOUT: {_tipo_nome} export did not finish in {PROCESSING_TIMEOUT_S}s")

log("========== DONE ==========")
for _f in downloaded_files:
    log(f"  downloaded: {_f}")

In [ ]:
# Persist the latest session and close the browser cleanly.
await context.storage_state(path=str(STATE_FILE))
await context.close()
await browser.close()
await _pw.stop()
log("Browser closed. Done.")

In [ ]:
# DEBUG HELPER (optional) -> run in headed mode to pause and inspect the live page
# with Playwright Inspector, e.g. to confirm a selector. Uncomment to use.
# await page.pause()